# CoTraffic : fine-tuning LoRA et prompt tuning du conseiller LLM

Ce notebook entraîne un **petit modèle** (Qwen2.5 0,5B ou 1,5B Instruct) à produire les
conseils du conseiller stratégique, de deux façons :

1. **LoRA** : de petites matrices de rang 16 ajoutées aux couches d'attention et du MLP.
   Quelques millions de paramètres entraînés, le modèle de base reste figé.
2. **Prompt tuning** : seulement 20 « jetons virtuels » appris devant le prompt. Environ
   20 000 paramètres, le modèle reste figé.

Il les compare au **prompt engineering** (le modèle de base avec le prompt zero-shot, sans
entraînement), sur les 63 situations de validation (graine 1000).

**Données** (créées par `scripts/llm_dataset.py`, dans `results/exploration/llm_lora/`) :
- `llm_lora_train.jsonl` : situations des **graines d'entraînement uniquement**
  (100001 à 100003). La réponse attendue est le meilleur conseil candidat trouvé par rejeu
  SUMO (règles, à la force 1, 2 ou 3), ou l'abstention si aucun candidat ne gagne au moins
  0,25 % ;
- `llm_lora_val.jsonl` : les 63 situations de validation, pour la mesure seulement.

**Durée** : environ 20 à 30 min avec 0,5B, 45 à 60 min avec 1,5B, sur un GPU T4 gratuit.

**Étapes sur Kaggle**
1. Settings : Accelerator **GPU T4 x2** (un seul est utilisé), **Internet on**.
2. Add Input > Upload : un dataset avec les deux fichiers `.jsonl` (lus dans `/kaggle/input`).
3. Run All. Les sorties (adaptateurs `.zip`, prédictions `.csv`) sont dans `/kaggle/working`,
   onglet Output.

**Étapes sur Colab** : Exécution > Modifier le type d'exécution > GPU T4, puis exécuter les
cellules dans l'ordre ; la cellule 3 demande les deux fichiers, la cellule 11 télécharge
les sorties.

Résultats **préliminaires** : peu d'exemples, et des étiquettes bruitées (le meilleur de
quelques candidats sur un rejeu de 5 min).

## 1. GPU (un seul : sur Kaggle, « T4 x2 » en donne deux, on n'en utilise qu'un)

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # before torch: one GPU, no DataParallel
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Activer un GPU : Exécution > Modifier le type d'exécution > T4"
print(torch.__version__, torch.cuda.get_device_name(0))
from pathlib import Path

## 2. Bibliothèques

In [ ]:
!pip -q install "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1"
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

## 3. Données

Sur **Colab**, la cellule demande les deux fichiers : `llm_lora_train.jsonl` et
`llm_lora_val.jsonl`. Sur **Kaggle**, les ajouter comme dataset (« Add data »), ils sont
alors lus dans `/kaggle/input/`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

NEEDED = ["llm_lora_train.jsonl", "llm_lora_val.jsonl"]
WORK = next((Path(d) for d in ("/kaggle/working", "/content") if Path(d).exists()), Path("."))
DATA = WORK / "data"
DATA.mkdir(parents=True, exist_ok=True)
for name in NEEDED:  # Kaggle: copy from the attached dataset
    for found in Path("/kaggle/input").glob(f"**/{name}") if Path("/kaggle/input").exists() else []:
        (DATA / name).write_bytes(found.read_bytes())
missing = [n for n in NEEDED if not (DATA / n).exists()]
if missing:
    from google.colab import files
    print("Choisir les fichiers :", ", ".join(missing))
    for name, content in files.upload().items():
        (DATA / name).write_bytes(content)

def read(name):
    return [json.loads(l) for l in (DATA / name).read_text(encoding="utf-8").splitlines() if l.strip()]

train, val = read("llm_lora_train.jsonl"), read("llm_lora_val.jsonl")
assert all(ex["split"] == "train" for ex in train) and all(ex["split"] == "val" for ex in val)
print(f"{len(train)} exemples d'entraînement, {len(val)} de validation")
print("étiquettes (entraînement) :", dict(Counter(ex["label"] for ex in train)))
print("étiquettes (validation)   :", dict(Counter(ex["label"] for ex in val)))

## 4. Réglages

`MODEL` : 0,5B par défaut (le plus rapide). **Si le 0,5B reste mauvais** (rappel des
conseils proche de 0), passer à `"Qwen/Qwen2.5-1.5B-Instruct"` : plus capable, environ
3 fois plus long (compter 30 à 45 min sur un T4), rien d'autre à changer.

Après le premier essai, où LoRA a appris à toujours s'abstenir : plus de pas
d'optimisation (`ACCUM` 2, 6 epochs pour LoRA, 10 pour le prompt tuning), et un
entraînement rééquilibré (cellule 5). Aucune valeur n'est réglée sur les graines de test.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # ou "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 0
MAX_LEN = 2048          # le prompt zero-shot fait au plus ~850 tokens, la réponse < 100
# LoRA
EPOCHS, LR, BATCH, ACCUM = 6, 2e-4, 2, 2   # 8 epochs possible if the loss still falls
R, ALPHA, DROPOUT = 16, 32, 0.05
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# prompt tuning
PT_TOKENS, PT_EPOCHS, PT_LR = 20, 10, 1e-2
PT_INIT = "You advise traffic signal agents. Answer with JSON advice, or an empty list."
# latency used when these answers are replayed locally (Ollama, same base model, seed 1000)
LOCAL_LATENCY_S = {"Qwen/Qwen2.5-0.5B-Instruct": 4.0, "Qwen/Qwen2.5-1.5B-Instruct": 4.8}[MODEL]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = WORK  # /kaggle/working on Kaggle, /content on Colab
torch.manual_seed(SEED)

## 5. Mise en forme et rééquilibrage

La perte ne porte que sur la réponse. Les abstentions sont majoritaires : sans
rééquilibrage, le modèle apprend à toujours s'abstenir (premier essai : 100 %
d'abstentions, et un accord de 87 % qui n'est que la part des abstentions dans la
validation). Les exemples de conseil sont donc **suréchantillonnés dans l'entraînement
seulement**, jusqu'à égaler les abstentions. La validation n'est jamais modifiée.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def prompt_text(ex):
    return tok.apply_chat_template(ex["messages"][:-1], tokenize=False, add_generation_prompt=True)

def encode(ex):
    p = tok(prompt_text(ex), add_special_tokens=False)["input_ids"]
    a = tok(ex["messages"][-1]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids, labels = (p + a)[-MAX_LEN:], ([-100] * len(p) + a)[-MAX_LEN:]
    return {"input_ids": ids, "labels": labels}

import math, random

def is_abstain(ex):
    return json.loads(ex["messages"][-1]["content"])["a"] == []

rng = random.Random(SEED)
abstain = [ex for ex in train if is_abstain(ex)]
advice = [ex for ex in train if not is_abstain(ex)]
extra = [advice[i % len(advice)] for i in range(max(0, len(abstain) - len(advice)))] if advice else []
balanced = abstain + advice + extra
rng.shuffle(balanced)
print(f"entraînement : {len(abstain)} abstentions, {len(advice)} conseils "
      f"(+{len(extra)} copies) -> {len(balanced)} exemples équilibrés")

train_enc = [encode(ex) for ex in balanced]
print("longueur max :", max(len(e["input_ids"]) for e in train_enc), "tokens")

def total_steps(epochs):
    return math.ceil(len(train_enc) / (BATCH * ACCUM)) * epochs

print(f"pas d'optimisation : LoRA {total_steps(EPOCHS)}, prompt tuning {total_steps(PT_EPOCHS)}")

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), n), PAD)
    labels = torch.full((len(batch), n), -100)
    mask = torch.zeros((len(batch), n), dtype=torch.long)
    for i, b in enumerate(batch):
        k = len(b["input_ids"])
        ids[i, :k], labels[i, :k], mask[i, :k] = torch.tensor(b["input_ids"]), torch.tensor(b["labels"]), 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}

## 6. Mesure sur la validation

Même lecture stricte que le projet (`src/cotraffic/llm/advice.py`) : JSON avec les seules
clés `a` et `r`, au plus 4 conseils `[carrefour, phase, force 1-3, durée 10-120 s]`.

| Mesure | Définition |
|---|---|
| JSON valide | Réponse lisible et dans les bornes |
| Abstention | Liste vide |
| Accord | Mêmes couples (carrefour, phase) que l'étiquette, ou abstention des deux côtés. **Trompeur seul** : « toujours s'abstenir » l'obtient pour la part des abstentions |
| Rappel des conseils | Sur les situations dont l'étiquette est un conseil : part des couples (carrefour, phase) de l'étiquette que le modèle propose |
| Faux conseils | Sur les situations dont l'étiquette est l'abstention : part où le modèle conseille quand même |

La référence triviale « toujours s'abstenir » est calculée à côté. Un modèle utile doit
battre son accord avec un rappel nettement au-dessus de 0. Le vrai juge reste le rejeu
dans SUMO (cellule 12). Les prédictions sont écrites au format du benchmark du projet.

Le modèle de base n'a pas de décodage contraint ici (Ollama, en local, impose le schéma
JSON) : son taux de JSON valide n'est donc pas comparable à celui du benchmark local.

In [ ]:
import csv, time

def parse_wire(text):
    try:
        d = json.loads(text)
    except ValueError:
        return None
    if not isinstance(d, dict) or set(d) - {"a", "r"} or len(str(d.get("r", ""))) > 60:
        return None
    items = d.get("a", [])
    if not isinstance(items, list) or len(items) > 4:
        return None
    out = []
    for it in items:
        if not (isinstance(it, list) and len(it) == 4):
            return None
        j, p, s, v = it
        ok = (isinstance(j, str) and type(p) is int and 0 <= p <= 7 and type(s) is int
              and s in (1, 2, 3) and type(v) is int and 10 <= v <= 120)
        if not ok:
            return None
        out.append((j, p, s, v))
    return out

from transformers import GenerationConfig

# greedy decoding: no temperature / top_p / top_k (avoids the sampling warnings)
GREEDY = GenerationConfig(
    do_sample=False, temperature=None, top_p=None, top_k=None, max_new_tokens=96,
    pad_token_id=PAD, eos_token_id=[tok.eos_token_id, tok.convert_tokens_to_ids("<|endoftext|>")],
)

def pairs(items):
    return {(j, p) for j, p, *_ in items}

def scores(preds, golds):
    # preds: parsed answers (None = invalid); golds: parsed labels
    n = len(golds)
    advice_gold = [(p, g) for p, g in zip(preds, golds) if g]
    abstain_gold = [p for p, g in zip(preds, golds) if not g]
    hit = sum(len(pairs(p or []) & pairs(g)) for p, g in advice_gold)
    need = sum(len(pairs(g)) for _, g in advice_gold)
    return {
        "json_valid_%": round(100 * sum(p is not None for p in preds) / n),
        "abstention_%": round(100 * sum(p == [] for p in preds) / n),
        "agreement_%": round(100 * sum(p is not None and pairs(p) == pairs(g) for p, g in zip(preds, golds)) / n),
        "advice_recall_%": round(100 * hit / need) if need else 0,
        "false_advice_%": round(100 * sum(bool(p) for p in abstain_gold) / len(abstain_gold)) if abstain_gold else 0,
    }

golds = [parse_wire(ex["messages"][-1]["content"]) for ex in val]
trivial = {"technique": "always_abstain", **scores([[] for _ in val], golds), "t4_latency_s": 0.0}
print("référence triviale :", trivial)

@torch.no_grad()
def evaluate(model, name):
    model.eval()
    rows, preds, latency_sum = [], [], 0.0
    for ex in val:
        enc = tok(prompt_text(ex), return_tensors="pt", add_special_tokens=False).to(model.device)
        start = time.perf_counter()
        out = model.generate(**enc, generation_config=GREEDY)
        latency = time.perf_counter() - start
        text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        pred = parse_wire(text)
        preds.append(pred)
        latency_sum += latency
        advice = {"a": [{"j": j, "p": p, "w": round(s / 3, 4), "s": v} for j, p, s, v in pred], "r": ""} if pred is not None else None
        rows.append({
            "model": f"{MODEL.split('/')[-1].lower()}-{name}", "digest": "", "technique": name,
            "prompt": "advisor-v2/zero", "situation": ex["id"], "kind": ex["kind"], "trigger": "",
            "latency_s": LOCAL_LATENCY_S, "attempts": 1, "valid": pred is not None,
            "first_valid": pred is not None, "items": len(pred or []),
            "advice": json.dumps(advice) if advice is not None else "", "prompt_tokens": enc["input_ids"].shape[1],
            "output_tokens": out.shape[1] - enc["input_ids"].shape[1], "error": "" if pred is not None else text[:200],
        })
    summary = {"technique": name, **scores(preds, golds), "t4_latency_s": round(latency_sum / len(val), 2)}
    with open(OUT / f"{name}_calls.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    with open(OUT / f"{name}_models.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["model", "digest", "size_bytes", "parameters", "quantization", "family", "license", "memory_gb", "technique", "prompt"])
        w.writeheader()
        w.writerow({"model": rows[0]["model"], "digest": "", "size_bytes": "", "parameters": MODEL.split("-")[1], "quantization": "fp16",
                    "family": "qwen2", "license": "Apache License", "memory_gb": "", "technique": name, "prompt": "advisor-v2/zero"})
    print(summary)
    return summary

## 7. Référence : le modèle de base, prompt zero-shot (prompt engineering seul)

In [ ]:
from transformers import AutoModelForCausalLM

def base_model():
    return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE)

results = [evaluate(base_model(), "base")]

## 8. Fine-tuning LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

def train(model, epochs, lr, out):
    model.config.use_cache = False
    args = TrainingArguments(
        output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=5, save_strategy="no", fp16=DEVICE == "cuda", report_to="none",
        seed=SEED, gradient_checkpointing=True, remove_unused_columns=False,
    )
    start = time.perf_counter()
    Trainer(model=model, args=args, train_dataset=train_enc, data_collator=collate).train()
    model.config.use_cache = True
    return round((time.perf_counter() - start) / 60, 1)

model = base_model()
model.enable_input_require_grads()
lora = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT,
                                        target_modules=TARGETS, task_type="CAUSAL_LM"))
lora.print_trainable_parameters()
lora_minutes = train(lora, EPOCHS, LR, str(OUT / "out_lora"))
lora.save_pretrained(str(OUT / "adapter_lora"))
summary = evaluate(lora, "lora")
summary.update(trainable_params=sum(p.numel() for p in lora.parameters() if p.requires_grad), train_minutes=lora_minutes)
results.append(summary)
del lora, model; torch.cuda.empty_cache()

## 9. Prompt tuning (soft prompt)

Seuls `PT_TOKENS` vecteurs d'embedding sont appris ; ils sont placés devant chaque prompt.
Initialisation à partir d'une phrase qui décrit la tâche.

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit

model = base_model()
pt = get_peft_model(model, PromptTuningConfig(
    task_type="CAUSAL_LM", prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=PT_INIT, num_virtual_tokens=PT_TOKENS, tokenizer_name_or_path=MODEL))
pt.print_trainable_parameters()
pt_minutes = train(pt, PT_EPOCHS, PT_LR, str(OUT / "out_pt"))
pt.save_pretrained(str(OUT / "adapter_prompt_tuning"))
summary = evaluate(pt, "prompt_tuning")
summary.update(trainable_params=sum(p.numel() for p in pt.parameters() if p.requires_grad), train_minutes=pt_minutes)
results.append(summary)

## 10. Comparaison

| Approche | Ce qui change | Paramètres entraînés | Coût |
|---|---|---|---|
| Prompt engineering (base) | Seulement le texte du prompt | 0 | Aucun entraînement ; le prompt est long (lu à chaque appel) |
| LoRA | Matrices de rang 16 dans chaque couche | Quelques millions (moins de 1 % du modèle) | Minutes sur un T4 ; un adaptateur de quelques dizaines de Mo |
| Prompt tuning | 20 vecteurs devant le prompt | Environ 20 000 | Le moins coûteux ; souvent moins efficace que LoRA sur les petits modèles |

Lire le tableau dans cet ordre : un modèle doit d'abord battre « toujours s'abstenir »
(même accord, mais un rappel des conseils au-dessus de 0 et peu de faux conseils). Même
alors, l'accord avec l'étiquette ne dit pas si les conseils améliorent le trafic : les
prédictions sont rejouées dans SUMO en local (dernière cellule).

In [ ]:
import pandas as pd
table = pd.DataFrame([trivial, *results]).fillna({"trainable_params": 0, "train_minutes": 0})
print(table.to_string(index=False))
table.to_csv(OUT / "lora_comparison.csv", index=False)

## 11. Téléchargement des résultats

In [ ]:
import shutil
for name in ("adapter_lora", "adapter_prompt_tuning"):
    shutil.make_archive(str(OUT / name), "zip", str(OUT / name))
outputs = ["adapter_lora.zip", "adapter_prompt_tuning.zip", "lora_comparison.csv",
           "base_calls.csv", "base_models.csv", "lora_calls.csv", "lora_models.csv",
           "prompt_tuning_calls.csv", "prompt_tuning_models.csv"]
try:
    from google.colab import files
    for name in outputs:
        files.download(str(OUT / name))
except ImportError:  # Kaggle: no browser download, files are in the Output panel
    print("Fichiers dans", OUT, "(onglet Output de Kaggle) :", outputs)

## 12. En local : rejouer les prédictions dans SUMO

Copier les fichiers `*_calls.csv` et `*_models.csv` dans
`results/exploration/llm_benchmark/`, puis, depuis `G:\pi5eme\cotraffic` :

```powershell
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls lora --techniques lora --betas 4 --tag lora_val
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls prompt_tuning --techniques prompt_tuning --betas 4 --tag pt_val
.venv\Scripts\python scripts\llm_benchmark.py --reuse-calls base --techniques base --betas 4 --tag base_val
```

Chaque conseil est rejoué 5 min depuis l'instantané de sa situation, après la latence
locale du même modèle de base sous Ollama. Le gain (positif = mieux) se compare à celui du
LLM zero-shot (+0,15 %) et à l'oracle (`results/exploration/llm_oracle/oracle_summary.csv`).

Pour utiliser l'adaptateur LoRA dans Ollama (facultatif) : le convertir en GGUF avec
`convert_lora_to_gguf.py` de llama.cpp, puis un `Modelfile` avec `FROM qwen2.5:0.5b` et
`ADAPTER ./adapter.gguf`.